In [24]:
!pip install -q -U "transformers>=4.57.0" "datasets>=3.0.0" "accelerate>=1.0.0" "peft>=0.15.2" "bitsandbytes>=0.44.0" "safetensors>=0.5.0" "torchao>=0.16.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 88.9 MB/s eta 0:00:00:00:01


In [25]:
from __future__ import annotations

import json
import math
import os
import random
import re
import shutil
from collections import Counter
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)


In [26]:
BASE_MODEL = "HuggingFaceTB/SmolLM3-3B"
OUTPUT_DIR = Path("/content/smollm3_answer_lora")

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Not running in Colab or Drive already mounted:', exc)

def find_repo_root():
    starts = [Path.cwd(), Path('/content'), Path('/content/til-26-overflow')]
    for start in starts:
        if not start.exists():
            continue
        for candidate in [start, *start.parents]:
            if (candidate / 'nlp' / 'src').exists() and ((candidate / 'CODEX.md').exists() or (candidate / 'data' / 'nlp').exists()):
                return candidate
    return Path('/content')

repo_root = find_repo_root()
shortcut_path = Path('/content/drive/MyDrive/til26_data')
drive_data_dir = shortcut_path / 'nlp'
repo_data_dir = repo_root / 'data' / 'nlp'
DATA_DIR = Path(os.getenv('NLP_DATA_DIR', drive_data_dir if (drive_data_dir / 'nlp.jsonl').exists() else repo_data_dir))
if not (DATA_DIR / 'nlp.jsonl').exists():
    raise FileNotFoundError(f'Could not find nlp.jsonl under {DATA_DIR}. Mount Drive or set NLP_DATA_DIR.')

artifact_dir = shortcut_path / 'nlp' / 'models' / 'smollm3_answer_model'
repo_artifact_dir = repo_root / 'nlp' / 'src' / 'models' / 'smollm3_answer_model'

MAX_CHUNK_WORDS = 170
CHUNK_OVERLAP_WORDS = 35
RETRIEVAL_TOP_K = 8
MAX_CONTEXT_CHARS = 3600
MAX_LENGTH = 4096

TRAIN_EPOCHS = float(os.getenv('TRAIN_EPOCHS', '2'))
LEARNING_RATE = float(os.getenv('LEARNING_RATE', '2e-4'))
BATCH_SIZE = int(os.getenv('BATCH_SIZE', '1'))
GRAD_ACCUM = int(os.getenv('GRAD_ACCUM', '8'))
MERGE_ADAPTER = os.getenv('MERGE_ADAPTER', '1') != '0'

print('repo_root:', repo_root)
print('data_dir:', DATA_DIR)
print('artifact_dir:', artifact_dir)
print('repo_artifact_dir:', repo_artifact_dir)
print('BASE_MODEL:', BASE_MODEL)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
repo_root: /content
data_dir: /content/drive/MyDrive/til26_data/nlp
artifact_dir: /content/drive/MyDrive/til26_data/nlp/models/smollm3_answer_model
repo_artifact_dir: /content/nlp/src/models/smollm3_answer_model
BASE_MODEL: HuggingFaceTB/SmolLM3-3B


In [27]:
TOKEN_RE = re.compile(r"\d+(?:[.,]\d+)?%?|[a-z0-9]+(?:[-'][a-z0-9]+)*", re.I)
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "did", "do", "does", "for",
    "from", "had", "has", "have", "how", "in", "into", "is", "it", "of", "on", "or",
    "the", "their", "to", "was", "were", "what", "when", "where", "which", "who",
    "whom", "whose", "why", "with",
}
EXPANSION_GROUPS = {
    "amount": {"amount", "budget", "cost", "costs", "funding", "much", "paid", "payment", "price", "revenue", "spend", "spent", "value", "valued", "worth"},
    "count": {"count", "many", "number", "quantity", "total"},
    "date": {"date", "day", "deadline", "month", "schedule", "time", "timeline", "when", "year"},
    "person": {"accountable", "ceo", "chair", "chief", "commander", "director", "founder", "head", "leader", "led", "manager", "officer", "person", "who"},
    "place": {"city", "country", "district", "facility", "location", "place", "region", "site", "where", "zone"},
    "negation": {"absent", "missing", "not", "noted", "omitted", "stated", "undisclosed", "unspecified"},
}

def normalize_space(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def tokens(text: str) -> list[str]:
    return [match.group(0).lower() for match in TOKEN_RE.finditer(text)]

def question_cue_groups(question: str) -> set[str]:
    lower = question.lower()
    terms = set(tokens(question))
    groups = set()
    if "how many" in lower:
        groups.add("count")
    if "how much" in lower:
        groups.add("amount")
    for phrase, group in (("when", "date"), ("where", "place"), ("who", "person"), ("whom", "person"), ("whose", "person")):
        if phrase in lower:
            groups.add(group)
    for group, group_terms in EXPANSION_GROUPS.items():
        if terms & group_terms:
            groups.add(group)
    if re.search(r"\b(no|not|never|without|missing|unstated|unmentioned)\b", lower):
        groups.add("negation")
    return groups

def expanded_query(question: str) -> str:
    parts = [question]
    for group in question_cue_groups(question):
        parts.extend(sorted(EXPANSION_GROUPS[group]))
    content_terms = [term for term in tokens(question) if term not in STOPWORDS]
    if content_terms:
        parts.append(" ".join(content_terms))
    return " ".join(parts)

In [28]:
def document_title(document: str, doc_id: int) -> str:
    for line in document.splitlines():
        line = normalize_space(line).strip("#* ")
        if line:
            return line[:180]
    return f"Document {doc_id + 1}"

def chunk_document(document: str, doc_id: int) -> list[dict]:
    document = document.replace("\r\n", "\n").replace("\r", "\n")
    title = document_title(document, doc_id)
    paragraphs = [normalize_space(part) for part in re.split(r"\n\s*\n", document) if normalize_space(part)]
    chunks = []
    current_words = []
    chunk_id = 0

    def flush():
        nonlocal chunk_id, current_words
        if not current_words:
            return
        body = " ".join(current_words)
        text = f"{title}\n{body}" if title and title not in body[:160] else body
        chunks.append({"doc_id": doc_id, "chunk_id": chunk_id, "title": title, "text": text})
        chunk_id += 1
        current_words = current_words[-CHUNK_OVERLAP_WORDS:] if CHUNK_OVERLAP_WORDS > 0 else []

    stride = max(1, MAX_CHUNK_WORDS - CHUNK_OVERLAP_WORDS)
    for paragraph in paragraphs:
        words = paragraph.split()
        if len(words) > MAX_CHUNK_WORDS:
            flush()
            for start in range(0, len(words), stride):
                window = words[start:start + MAX_CHUNK_WORDS]
                if window:
                    chunks.append({"doc_id": doc_id, "chunk_id": chunk_id, "title": title, "text": f"{title}\n{' '.join(window)}"})
                    chunk_id += 1
            current_words = []
            continue
        if current_words and len(current_words) + len(words) > MAX_CHUNK_WORDS:
            flush()
        current_words.extend(words)
    flush()
    return chunks

documents = []
for path in sorted((DATA_DIR / "documents").glob("*.txt")):
    documents.append(path.read_text(encoding="utf-8"))

chunks = []
for doc_id, document in enumerate(documents):
    chunks.extend(chunk_document(document, doc_id))

doc_freq = Counter()
chunk_terms = []
chunk_lengths = []
for chunk in chunks:
    terms = Counter(tokens(chunk["text"]))
    chunk_terms.append(terms)
    chunk_lengths.append(sum(terms.values()))
    doc_freq.update(terms.keys())

avgdl = sum(chunk_lengths) / max(1, len(chunk_lengths))
idf = {term: math.log(1.0 + (len(chunks) - freq + 0.5) / (freq + 0.5)) for term, freq in doc_freq.items()}
print(f"Loaded {len(documents)} documents and {len(chunks)} chunks")

Loaded 296 documents and 2555 chunks


In [29]:
def retrieve_bm25(question: str, k: int = RETRIEVAL_TOP_K) -> list[dict]:
    query_terms = Counter(tokens(expanded_query(question)))
    if not query_terms:
        return []
    scores = []
    k1 = 1.45
    b = 0.72
    for index, terms in enumerate(chunk_terms):
        score = 0.0
        length = chunk_lengths[index] or 1
        length_norm = k1 * (1.0 - b + b * length / avgdl)
        for term, query_count in query_terms.items():
            frequency = terms.get(term, 0)
            if not frequency:
                continue
            weight = 0.35 if term in STOPWORDS else 1.0
            tf = (frequency * (k1 + 1.0)) / (frequency + length_norm)
            score += idf.get(term, 0.0) * tf * weight * min(2.0, query_count)
        if score > 0:
            scores.append((index, score))
    scores.sort(key=lambda item: item[1], reverse=True)
    return [chunks[index] for index, _score in scores[:k]]

def build_context(question: str) -> str:
    blocks = []
    used_chars = 0
    for evidence_index, chunk in enumerate(retrieve_bm25(question), start=1):
        block = normalize_space(chunk["text"])
        block = f"[{evidence_index}] {chunk['title']}: {block}"
        if used_chars + len(block) > MAX_CONTEXT_CHARS:
            remaining = MAX_CONTEXT_CHARS - used_chars
            if remaining <= 240:
                break
            block = block[:remaining].rsplit(" ", 1)[0]
        blocks.append(block)
        used_chars += len(block)
    return "\n".join(blocks)

In [30]:
instances = []
with (DATA_DIR / "nlp.jsonl").open("r", encoding="utf-8") as handle:
    for line in handle:
        if line.strip():
            instances.append(json.loads(line))

SYSTEM_PROMPT = (
    "/no_think\n"
    "You answer retrieval questions using only the provided evidence. "
    "Return one concise answer string. Do not explain. "
    "For arithmetic or comparisons, compute the answer from the evidence. "
    "For answerable questions, do not return empty text. "
    "Do not mention the evidence or context."
)

def make_training_row(instance: dict) -> dict:
    question = instance["question"].strip()
    answer = instance.get("answer")
    if answer is None:
        answer = ""
    context = build_context(question)
    user_prompt = (
        f"Question: {question}\n\n"
        f"Evidence:\n{context}\n\n"
        "Answer with the shortest phrase or sentence that fully answers the question:"
    )
    return {"question": question, "answer": str(answer).strip(), "user_prompt": user_prompt}

rows = [make_training_row(instance) for instance in instances]
random.shuffle(rows)
split = max(1, int(0.9 * len(rows)))
train_rows = rows[:split]
eval_rows = rows[split:]
print(f"Train rows: {len(train_rows)} | Eval rows: {len(eval_rows)}")
print(train_rows[0]["user_prompt"][:1000])
print("ANSWER:", train_rows[0]["answer"])

Train rows: 794 | Eval rows: 89
Question: Under what CGC Entertainment Venue License Class and permit number is Caulfield's Casino registered?

Evidence:
[1] CLAIROS GOVERNANCE COUNCIL: CLAIROS GOVERNANCE COUNCIL Haven Island Special Economic Zone to file renewal documentation no later than the twenty-second day of the first month of each PCE licensing year. This submission is timely filed. --- ## SECTION 2: VENUE IDENTIFICATION | Field | Details | |---|---| | **Registered Venue Name** | Caulfield's Casino and Entertainment Venue | | **Registered Address** | 44 Meridian Row, Downtown South Haven, Haven Island Special Economic Zone | | **Venue Classification** | Class III Entertainment Venue (Gaming, Hospitality, and Licensed Alcohol Service) | | **Permit Number** | SH-EV-00714 | | **Sector Designation** | South Haven | | **Permitted Operating Hours** | 06:00–04:00, seven days per week, per standard Class III authorization | --- ## SECTION 3: REGISTERED PROPRIETORSHIP AND MANAGEMENT
[2]

In [31]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def chat_prompt(user_prompt: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def tokenize_row(row: dict) -> dict:
    prompt = chat_prompt(row["user_prompt"])
    answer = row["answer"] + (tokenizer.eos_token or "")
    full_text = prompt + answer
    encoded = tokenizer(full_text, max_length=MAX_LENGTH, truncation=True)
    prompt_ids = tokenizer(prompt, max_length=MAX_LENGTH, truncation=True)["input_ids"]
    labels = encoded["input_ids"].copy()
    prompt_len = min(len(prompt_ids), len(labels))
    labels[:prompt_len] = [-100] * prompt_len
    encoded["labels"] = labels
    return encoded

train_dataset = Dataset.from_list(train_rows).map(tokenize_row, remove_columns=list(train_rows[0].keys()))
eval_dataset = Dataset.from_list(eval_rows).map(tokenize_row, remove_columns=list(eval_rows[0].keys())) if eval_rows else None
print(train_dataset[0].keys())

Map:   0%|          | 0/794 [00:00<?, ? examples/s]

Map:   0%|          | 0/89 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quantization_config,
    device_map="auto",
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=TRAIN_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch" if eval_dataset is not None and len(eval_dataset) else "no",
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
)

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
)
trainer.train()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

In [32]:
def get_preview_model():
    preview_model = globals().get("model") or globals().get("preview_model") or globals().get("merged_model")
    if preview_model is not None:
        preview_model.eval()
        return preview_model

    adapter_dir = Path(OUTPUT_DIR)
    if not (adapter_dir / "adapter_config.json").exists():
        raise FileNotFoundError(f"No live model variable and no LoRA adapter found at {adapter_dir}")

    import importlib.metadata
    import subprocess
    import sys
    try:
        torchao_version = importlib.metadata.version("torchao")
        major, minor, *_ = [int(part) for part in torchao_version.split(".")[:2]]
        if (major, minor) < (0, 16):
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "torchao>=0.16.0"])
            for module_name in list(sys.modules):
                if module_name == "torchao" or module_name.startswith("torchao."):
                    sys.modules.pop(module_name, None)
            print("Updated torchao for PEFT adapter loading.")
    except importlib.metadata.PackageNotFoundError:
        pass

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=quantization_config,
        device_map="auto",
    )
    preview_model = PeftModel.from_pretrained(base_model, str(adapter_dir))
    preview_model.eval()
    globals()["preview_model"] = preview_model
    return preview_model


def generate_answer(question: str) -> str:
    context = build_context(question)
    user_prompt = (
        f"Question: {question}\n\n"
        f"Evidence:\n{context}\n\n"
        "Answer with the shortest phrase or sentence that fully answers the question:"
    )
    prompt = chat_prompt(user_prompt)
    preview_model = get_preview_model()
    device = next(preview_model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = preview_model.generate(
            **inputs,
            max_new_tokens=96,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    answer_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(answer_ids, skip_special_tokens=True).strip()

for row in eval_rows[:5] if eval_rows else train_rows[:5]:
    print("Q:", row["question"])
    print("GT:", row["answer"])
    print("PRED:", generate_answer(row["question"]))
    print("---")

Q: How many full reset cycles of the nanobot adaptation algorithm would a single uninterrupted minute of rapid Edge/baseline oscillation force, given the algorithm's calibration interval?
GT: The source facts do not specify the oscillation frequency during rapid alternation, so the number of full reset cycles per minute cannot be determined from the given information alone. The only stated figure is the 4.7-second calibration cycle; without knowing how often oscillations occur, the total reset cycles in one minute is indeterminate.


Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

PRED: Approximately 1.7 cycles per minute
---
Q: How many new Conclave adherents did the CGC estimate were recruited within Haven during the 76-77 PCE period?
GT: approximately 280
PRED: approximately 280
---
Q: How many people make up the internal team authorized by Park Soo-Hyun for confidential partnership exploration under Project Crescendo?
GT: Three
PRED: Three
---
Q: What type of communities does Ashcastle call for in the 69 PCE broadcast?
GT: 'sanctuaries of the unaltered'
PRED: Opted communities
---
Q: How did the CGC officially characterize the events at the 25th anniversary ceremony in its public statement?
GT: 'security incident'
PRED: 'a security incident'
---


In [33]:
if MERGE_ADAPTER:
    if "model" in globals():
        del model
    torch.cuda.empty_cache()
    import importlib.metadata
    import subprocess
    import sys
    try:
        torchao_version = importlib.metadata.version("torchao")
        major, minor, *_ = [int(part) for part in torchao_version.split(".")[:2]]
        if (major, minor) < (0, 16):
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "torchao>=0.16.0"])
            for module_name in list(sys.modules):
                if module_name == "torchao" or module_name.startswith("torchao."):
                    sys.modules.pop(module_name, None)
            print("Updated torchao for PEFT adapter merge.")
    except importlib.metadata.PackageNotFoundError:
        pass
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16,
        device_map="auto",
    )
    merged_model = PeftModel.from_pretrained(base_model, str(OUTPUT_DIR))
    merged_model = merged_model.merge_and_unload()
    artifact_dir.parent.mkdir(parents=True, exist_ok=True)
    if artifact_dir.exists():
        shutil.rmtree(artifact_dir)
    merged_model.config.use_cache = True
    merged_model.save_pretrained(str(artifact_dir), safe_serialization=True)
    tokenizer.save_pretrained(str(artifact_dir))

    if (repo_root / 'nlp' / 'src').exists():
        repo_artifact_dir.parent.mkdir(parents=True, exist_ok=True)
        if repo_artifact_dir.exists():
            shutil.rmtree(repo_artifact_dir)
        shutil.copytree(artifact_dir, repo_artifact_dir)

    zip_base = artifact_dir.parent / 'smollm3_answer_model'
    shutil.make_archive(str(zip_base), 'zip', artifact_dir)

    print('Saved model folder:', artifact_dir)
    print('Saved repo copy:', repo_artifact_dir)
    print('Saved zip:', str(zip_base) + '.zip')
else:
    artifact_dir.parent.mkdir(parents=True, exist_ok=True)
    zip_base = artifact_dir.parent / 'smollm3_answer_lora'
    shutil.make_archive(str(zip_base), 'zip', OUTPUT_DIR)
    print('Adapter saved to:', OUTPUT_DIR)
    print('Adapter zip:', str(zip_base) + '.zip')


Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model folder: /content/drive/MyDrive/til26_data/nlp/models/smollm3_answer_model
Saved repo copy: /content/nlp/src/models/smollm3_answer_model
Saved zip: /content/drive/MyDrive/til26_data/nlp/models/smollm3_answer_model.zip


In [37]:
!ls drive/MyDrive/til26_data/nlp/models

nlp_answer_model      qwen_answer_model      smollm3_answer_model.zip
nlp_answer_model.zip  qwen_answer_model.zip
nlp_eval.zip	      smollm3_answer_model


In [38]:
!ls -lh /content/drive/MyDrive/til26_data/nlp/models/smollm3_answer_model.zip


-rw------- 1 root root 4.6G May 13 13:39 /content/drive/MyDrive/til26_data/nlp/models/smollm3_answer_model.zip
